In [1]:
# ============================================================
# PHASE 2 — LABEL DEFINITION
# ============================================================
import os, json
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'
SPLITS_DIR = os.path.join(PROJECT_DIR, 'splits')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# ============================================================
# DECISION: Choose label protocol
# ============================================================
# Option A: "N_V_Q" — True 3-class (S excluded)
# Option B: "NS_V_Q" — S explicitly merged with N
#
# Recommendation: Option B (NS_V_Q) — clinically valid
# Reason: S (supraventricular) and N (normal) both have narrow QRS
#         Single-lead ECG discrimination is hard (as we saw: S F1 ~10-20%)

LABEL_PROTOCOL = "NS_V_Q"  # <-- Change to "N_V_Q" if you want true N/V/Q

print(f"Selected Label Protocol: {LABEL_PROTOCOL}")

# ============================================================
# Define label mapping
# ============================================================
if LABEL_PROTOCOL == "N_V_Q":
    # True N/V/Q (S excluded)
    KEEP_CLASSES = [0, 2, 4]  # N, V, Q
    LABEL_MAP = {0: 0, 2: 1, 4: 2}
    CLASS_NAMES = ['N', 'V', 'Q']
    print("Classes: N (Normal), V (Ventricular), Q (Paced)")
    print("S-class: EXCLUDED")

elif LABEL_PROTOCOL == "NS_V_Q":
    # NS merged with N
    KEEP_CLASSES = [0, 1, 2, 4]  # N, S, V, Q (F excluded)
    LABEL_MAP = {0: 0, 1: 0, 2: 1, 4: 2}  # N,S → 0; V → 1; Q → 2
    CLASS_NAMES = ['N/S', 'V', 'Q']
    print("Classes: N/S (Normal+Supraventricular), V (Ventricular), Q (Paced)")
    print("S-class: MERGED into N/S explicitly")

# ============================================================
# Save label protocol config
# ============================================================
label_config = {
    'label_protocol': LABEL_PROTOCOL,
    'classes': CLASS_NAMES,
    'keep_classes': KEEP_CLASSES,
    'label_map': LABEL_MAP,
    'notes': {
        'N_V_Q': 'True 3-class; S-class excluded (insufficient discriminability)',
        'NS_V_Q': 'S merged with N; clinically valid (both narrow QRS)',
    }[LABEL_PROTOCOL]
}

config_path = os.path.join(OUTPUTS_DIR, 'label_config.json')
with open(config_path, 'w') as f:
    json.dump(label_config, f, indent=2)

print(f"\n✅ Label config saved: {config_path}")

# ============================================================
# Apply to current data (test with old NPZ)
# ============================================================
PROCESSED_DIR = os.path.join(PROJECT_DIR, 'data', 'processed')

# Load old data (from previous Step 2)
try:
    train = np.load(os.path.join(PROCESSED_DIR, 'train.npz'), allow_pickle=True)
    y_train = train['y']

    print(f"\nOriginal train labels: {dict(zip(*np.unique(y_train, return_counts=True)))}")

    # Apply mapping
    if LABEL_PROTOCOL == "N_V_Q":
        mask = np.isin(y_train, KEEP_CLASSES)
        y_train_new = y_train[mask].copy()
        for old, new in LABEL_MAP.items():
            y_train_new[y_train_new == old] = new
    else:  # NS_V_Q
        mask = np.isin(y_train, KEEP_CLASSES)
        y_train_new = y_train[mask].copy()
        y_train_new[y_train_new == 1] = 0  # S → N
        y_train_new[y_train_new == 2] = 1  # V → 1
        y_train_new[y_train_new == 4] = 2  # Q → 2

    print(f"\nNew train labels: {dict(zip(*np.unique(y_train_new, return_counts=True)))}")
    print(f"\nClass names: {CLASS_NAMES}")

except Exception as e:
    print(f"Note: Old NPZ not found or error: {e}")
    print("(Will be applied in Phase 3 — Step 2 rebuild)")

print("\n" + "=" * 70)
print("PHASE 2 COMPLETE — LABEL DEFINITION")
print("=" * 70)
print(f"Protocol: {LABEL_PROTOCOL}")
print(f"Classes: {CLASS_NAMES}")
print(f"Config saved: {config_path}")
print("\n" + "=" * 70)
print("NEXT: Phase 3 — Step 2 Rebuild")
print("=" * 70)

Selected Label Protocol: NS_V_Q
Classes: N/S (Normal+Supraventricular), V (Ventricular), Q (Paced)
S-class: MERGED into N/S explicitly

✅ Label config saved: /content/drive/MyDrive/ecg-transcovnet/outputs/label_config.json
Note: Old NPZ not found or error: [Errno 2] No such file or directory: '/content/drive/MyDrive/ecg-transcovnet/data/processed/train.npz'
(Will be applied in Phase 3 — Step 2 rebuild)

PHASE 2 COMPLETE — LABEL DEFINITION
Protocol: NS_V_Q
Classes: ['N/S', 'V', 'Q']
Config saved: /content/drive/MyDrive/ecg-transcovnet/outputs/label_config.json

NEXT: Phase 3 — Step 2 Rebuild


In [2]:
import os
PROCESSED_DIR = '/content/drive/MyDrive/ecg-transcovnet/data/processed'

print(f"Contents of {PROCESSED_DIR}:")
if os.path.exists(PROCESSED_DIR):
    for f in os.listdir(PROCESSED_DIR):
        size = os.path.getsize(os.path.join(PROCESSED_DIR, f)) / 1024 / 1024
        print(f"  {f}: {size:.2f} MB")
else:
    print(f"  ❌ Directory not found")

Contents of /content/drive/MyDrive/ecg-transcovnet/data/processed:
  ❌ Directory not found


In [3]:
import os

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

print("=" * 70)
print("DRIVE STRUCTURE CHECK")
print("=" * 70)

def check_dir(path, indent=0):
    prefix = "  " * indent
    if os.path.exists(path):
        items = os.listdir(path)
        print(f"{prefix}📁 {os.path.basename(path)}/ ({len(items)} items)")
        for item in sorted(items)[:10]:
            item_path = os.path.join(path, item)
            if os.path.isfile(item_path):
                size = os.path.getsize(item_path) / 1024 / 1024
                print(f"{prefix}   📄 {item}: {size:.2f} MB")
            else:
                check_dir(item_path, indent + 1)
        if len(items) > 10:
            print(f"{prefix}   ... and {len(items) - 10} more")
    else:
        print(f"{prefix}❌ {path} NOT FOUND")

check_dir(PROJECT_DIR)

DRIVE STRUCTURE CHECK
📁 ecg-transcovnet/ (1 items)
  📁 outputs/ (1 items)
     📄 label_config.json: 0.00 MB


In [4]:
# ============================================================
# FORCE REMOUNT DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

print("=" * 70)
print("DRIVE CONTENTS AFTER REMOUNT")
print("=" * 70)

for item in sorted(os.listdir(PROJECT_DIR)):
    item_path = os.path.join(PROJECT_DIR, item)
    if os.path.isdir(item_path):
        count = len(os.listdir(item_path))
        print(f"📁 {item}/ ({count} items)")
    else:
        size = os.path.getsize(item_path) / 1024 / 1024
        print(f"📄 {item}: {size:.2f} MB")

ValueError: Mountpoint must not already contain files

In [5]:
# ============================================================
# UNMOUNT + REMOUNT DRIVE
# ============================================================
from google.colab import drive
drive.flush_and_unmount()
print("✅ Drive unmounted")

Drive not mounted, so nothing to flush and unmount.
✅ Drive unmounted


In [6]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive re-mounted")

ValueError: Mountpoint must not already contain files

In [1]:
# ============================================================
# FRESH DRIVE MOUNT (after restart)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

print("=" * 70)
print("DRIVE CONTENTS AFTER FRESH MOUNT")
print("=" * 70)

if os.path.exists(PROJECT_DIR):
    for item in sorted(os.listdir(PROJECT_DIR)):
        item_path = os.path.join(PROJECT_DIR, item)
        if os.path.isdir(item_path):
            count = len(os.listdir(item_path))
            print(f"📁 {item}/ ({count} items)")
        else:
            size = os.path.getsize(item_path) / 1024 / 1024
            print(f"📄 {item}: {size:.2f} MB")
else:
    print(f"❌ {PROJECT_DIR} NOT FOUND")

Mounted at /content/drive
DRIVE CONTENTS AFTER FRESH MOUNT
📁 audit/ (4 items)
📁 backups/ (4 items)
📁 data/ (3 items)
📁 models/ (6 items)
📁 outputs/ (9 items)
📁 splits/ (2 items)


In [2]:
# ============================================================
# DEEP DRIVE CHECK — All folders
# ============================================================
import os

PROJECT_DIR = '/content/drive/MyDrive/ecg-transcovnet'

print("=" * 70)
print("DEEP DRIVE CHECK")
print("=" * 70)

folders_to_check = ['audit', 'backups', 'data', 'models', 'outputs', 'splits']

for folder in folders_to_check:
    path = os.path.join(PROJECT_DIR, folder)
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"\n📁 {folder}/ ({len(files)} items)")
        for f in sorted(files)[:10]:
            f_path = os.path.join(path, f)
            if os.path.isfile(f_path):
                size = os.path.getsize(f_path) / 1024 / 1024
                print(f"   📄 {f}: {size:.2f} MB")
            else:
                sub_count = len(os.listdir(f_path))
                print(f"   📁 {f}/ ({sub_count} items)")
        if len(files) > 10:
            print(f"   ... and {len(files) - 10} more")

# CRITICAL: Deep check on data/
print("\n" + "=" * 70)
print("CRITICAL: DATA FOLDER DEEP CHECK")
print("=" * 70)

data_path = os.path.join(PROJECT_DIR, 'data')
for item in sorted(os.listdir(data_path)):
    item_path = os.path.join(data_path, item)
    if os.path.isdir(item_path):
        sub_count = len(os.listdir(item_path))
        print(f"\n📁 data/{item}/ ({sub_count} items)")
        # Show first 10
        for f in sorted(os.listdir(item_path))[:10]:
            f_path = os.path.join(item_path, f)
            if os.path.isfile(f_path):
                size = os.path.getsize(f_path) / 1024 / 1024
                print(f"   📄 {f}: {size:.2f} MB")
        if sub_count > 10:
            print(f"   ... and {sub_count - 10} more")

DEEP DRIVE CHECK

📁 audit/ (4 items)
   📄 CURRENT_PIPELINE_AUDIT.md: 0.00 MB
   📄 label_audit.json: 0.00 MB
   📄 leakage_audit.json: 0.00 MB
   📄 subject_mapping.json: 0.00 MB

📁 backups/ (4 items)
   📄 models_20260917_050134.zip: 0.00 MB
   📄 outputs_20260917_050134.zip: 0.00 MB
   📄 processed_20260917_050134.zip: 93.83 MB
   📄 splits_20260917_050134.zip: 0.00 MB

📁 data/ (3 items)
   📁 mitdb/ (144 items)
   📁 processed/ (4 items)
   📁 raw/ (1 items)

📁 models/ (6 items)
   📄 baseline_cnn_3class_augmented.keras: 1.88 MB
   📄 baseline_cnn_3class_final.keras: 1.88 MB
   📄 baseline_cnn_3class_tuned.keras: 1.88 MB
   📄 baseline_cnn_3class_v_tuned.keras: 1.88 MB
   📄 baseline_cnn_4class_final.keras: 1.88 MB
   📄 pan_tompkins_hybrid.py: 0.00 MB

📁 outputs/ (9 items)
   📄 baseline_cnn_3class_5fold.json: 0.00 MB
   📄 baseline_cnn_4class_5fold.json: 0.00 MB
   📄 baseline_cnn_5fold_cv.json: 0.00 MB
   📄 oof_predictions_best.npy: 0.44 MB
   📄 oof_y_true.npy: 0.44 MB
   📄 pan_tompkins_hybrid_resu